# 1.2 Weather Data Fetching

In this notebook we fetch the weather data from archive-api.open-meteo.com (downloaded on 28.05.2026).

In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Import packages                         #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import pandas as pd

import requests

# weather data
import openmeteo_requests
import requests_cache
from retry_requests import retry

import time

import os

In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Reset working directory                 #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

# Depending on where the ipython kernel was started, we need to switch directories
if os.path.basename(os.getcwd()) != 'my_project':
    print(f'Dir: {os.getcwd()}')
    print('Going up a level...')
    os.chdir('..')
    print(f'Dir: {os.getcwd()}')
else:
    print(f'Already at project root: {os.getcwd()}')

In [2]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Fetch weather data					  #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
cache_session.cache.clear()
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": 41.8781,
	"longitude": -87.6298,
	"start_date": "2024-01-01",
	"end_date": "2026-01-01",
	"daily": ["sunrise", "sunset", "daylight_duration", "sunshine_duration"],
	"hourly": ["temperature_2m", "relative_humidity_2m", "apparent_temperature", "precipitation", "rain", "snowfall", "snow_depth", "surface_pressure", "cloud_cover", "wind_speed_10m", "wind_speed_100m", "is_day", "sunshine_duration", "direct_radiation"],
	"timezone": "UTC",
}
responses = openmeteo.weather_api(url, params=params)

response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
hourly_apparent_temperature = hourly.Variables(2).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(3).ValuesAsNumpy()
hourly_rain = hourly.Variables(4).ValuesAsNumpy()
hourly_snowfall = hourly.Variables(5).ValuesAsNumpy()
hourly_snow_depth = hourly.Variables(6).ValuesAsNumpy()
hourly_surface_pressure = hourly.Variables(7).ValuesAsNumpy()
hourly_cloud_cover = hourly.Variables(8).ValuesAsNumpy()
hourly_wind_speed_10m = hourly.Variables(9).ValuesAsNumpy()
hourly_wind_speed_100m = hourly.Variables(10).ValuesAsNumpy()
hourly_is_day = hourly.Variables(11).ValuesAsNumpy()
hourly_sunshine_duration = hourly.Variables(12).ValuesAsNumpy()
hourly_direct_radiation = hourly.Variables(13).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
	end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
hourly_data["apparent_temperature"] = hourly_apparent_temperature
hourly_data["precipitation"] = hourly_precipitation
hourly_data["rain"] = hourly_rain
hourly_data["snowfall"] = hourly_snowfall
hourly_data["snow_depth"] = hourly_snow_depth
hourly_data["surface_pressure"] = hourly_surface_pressure
hourly_data["cloud_cover"] = hourly_cloud_cover
hourly_data["wind_speed_10m"] = hourly_wind_speed_10m
hourly_data["wind_speed_100m"] = hourly_wind_speed_100m
hourly_data["is_day"] = hourly_is_day
hourly_data["sunshine_duration"] = hourly_sunshine_duration
hourly_data["direct_radiation"] = hourly_direct_radiation

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)

daily = response.Daily()
daily_sunrise = daily.Variables(0).ValuesInt64AsNumpy()
daily_sunset = daily.Variables(1).ValuesInt64AsNumpy()
daily_daylight_duration = daily.Variables(2).ValuesAsNumpy()
daily_sunshine_duration = daily.Variables(3).ValuesAsNumpy()

daily_data = {"date": pd.date_range(
	start = pd.to_datetime(daily.Time(), unit = "s", utc = True),
	end =  pd.to_datetime(daily.TimeEnd(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = daily.Interval()),
	inclusive = "left"
)}

daily_data["sunrise"] = daily_sunrise
daily_data["sunset"] = daily_sunset
daily_data["daylight_duration"] = daily_daylight_duration
daily_data["sunshine_duration"] = daily_sunshine_duration

daily_dataframe = pd.DataFrame(data = daily_data)
print("\nDaily data\n", daily_dataframe)

Coordinates: 41.8629150390625°N -87.64877319335938°E
Elevation: 241.0 m asl
Timezone: NoneNone
Timezone difference to GMT+0: 0s

Hourly data
                            date  temperature_2m  relative_humidity_2m  \
0     2024-01-01 00:00:00+00:00            0.80             80.643494   
1     2024-01-01 01:00:00+00:00            0.45             81.492714   
2     2024-01-01 02:00:00+00:00            0.20             81.760498   
3     2024-01-01 03:00:00+00:00            0.20             81.760498   
4     2024-01-01 04:00:00+00:00            0.25             81.767311   
...                         ...             ...                   ...   
17563 2026-01-01 19:00:00+00:00           -8.60             64.735161   
17564 2026-01-01 20:00:00+00:00           -8.45             65.036278   
17565 2026-01-01 21:00:00+00:00           -8.15             63.796635   
17566 2026-01-01 22:00:00+00:00           -8.10             64.330788   
17567 2026-01-01 23:00:00+00:00           -8.10        

In [3]:
daily_dataframe.head()

,date,sunrise,sunset,daylight_duration,sunshine_duration
0,2024-01-01 00:00:00+00:00,1704115086,1704148190,33092.765625,29168.417969
1,2024-01-02 00:00:00+00:00,1704201492,1704234643,33139.140625,23716.412109
2,2024-01-03 00:00:00+00:00,1704287895,1704321097,33189.378906,6035.676270
3,2024-01-04 00:00:00+00:00,1704374296,1704407553,33243.394531,29447.990234
4,2024-01-05 00:00:00+00:00,1704460694,1704494010,33301.089844,25576.648438


In [4]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Permanently Store Weather Data          #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

# convert timestamps to datetime to avoid false interpretation of dates
daily_dataframe["date"] = pd.to_datetime(daily_dataframe["date"], utc=True)
hourly_dataframe["date"] = pd.to_datetime(hourly_dataframe["date"], utc=True)

# convert sunrise and sunset times from iso8601 to datetime
daily_dataframe["sunrise"] = pd.to_datetime(daily_dataframe["sunrise"], unit="s", utc=True)
daily_dataframe["sunset"] = pd.to_datetime(daily_dataframe["sunset"], unit="s", utc=True)

daily_dataframe.to_csv("../../data/weather/weather_daily.csv", index=False)
hourly_dataframe.to_csv("../../data/weather/weather_hourly.csv", index=False)